# Prepare files for different analysis

In [1]:
import reframed
from reframed import SteadyCom
import pandas as pd
import numpy as np
from os import listdir
import warnings
import seaborn as sns
from collections import OrderedDict
import json

##  Prepare files for SMETANA 

We need:
- Community models
- Community data file

We also need:
- Media

#### Load data and models

**Load relative abundance (E. coli)**

In [2]:
relative_abundance_ecoli = pd.read_csv("../data/Guillen2021/relative_abundance.csv",index_col=0)
relative_abundance_ecoli.index = relative_abundance_ecoli.index.map(lambda x:x if x!="yhhK" else "panZ")

**Load models**

*E. coli mutants  (top 7)*

In [3]:
RA_ecoli_mean = relative_abundance_ecoli.T.mean().sort_values(ascending=False)
top7 = RA_ecoli_mean[RA_ecoli_mean>0.01].index
top7_string = [f"mutant_{mutant}.xml" for mutant in top7]

In [4]:
models_ecoli = {}
with warnings.catch_warnings():
    warnings.filterwarnings('ignore')
    for path in listdir("../models/Guillen2021/strains/"):
        if path not in top7_string: continue
        models_ecoli[path.split("_")[1][:-4]] = reframed.load_cbmodel("../models/Guillen2021/strains/"+path,
                                                        exchange_detection="^R_EX_(?!nad_c$|thf_c$).*")

*Gut microbiome members*

In [5]:
models_gut = {}
with warnings.catch_warnings():
    warnings.filterwarnings('ignore')
    for path in listdir("../models/Venturelli2018/species_used_noO2/"):
        models_gut[path[:-4]] = reframed.load_cbmodel("../models/Venturelli2018/species_used_noO2/"+path,
                                                      exchange_detection="R_EX_(?!biomass__40__e__41__)")


**Load media**

*For E. coli mutants (M9 medium to force growth of specific members)*

In [6]:
media_db = pd.read_csv("https://raw.githubusercontent.com/cdanielmachado/carveme/refs/heads/master/carveme/data/input/media_db.tsv",sep="\t")
compounds = list(media_db[media_db.medium=="M9"].compound.values)
env_ecoli = reframed.Environment.from_compounds(compounds,max_uptake=10)

*For gut microbiome members*

In [7]:
media_db_gut = pd.read_csv("../data/Venturelli2018/generated/media_db.tsv",sep="\t")
reactions = media_db_gut[media_db_gut.medium=="LB_extend"].compound.values
#reactions = [rxn.replace("40","91").replace("41","93") for rxn in reactions]

env_gut = reframed.Environment.from_reactions(reactions,max_uptake=10)

**Load other data**

In [8]:
gene_id2name = pd.read_csv("../data/Guillen2021/generated/gene_id2name.tsv",sep="\t")

#### E.coli mutants - Prepare files

In [9]:
model_ids = [model.id for model in models_ecoli.values()]
community_ids = ["ecoli_auxotrophs" for model in models_ecoli.values()]

pd.DataFrame([community_ids,model_ids]).T.to_csv("../data/Guillen2021/generated/communities_smetana.tsv",
                                        index=None,
                                        header=None,
                                        sep="\t")

#### Gut microbiome - Prepare files

In [10]:
df_comms = []

# Full community
model_ids = [model_id for model_id in models_gut.keys()]
community_ids = ["Full" for model_id in model_ids]
df_comm = pd.DataFrame([community_ids,model_ids])
df_comms.append(df_comm)

# Reduced communities
for model_id in models_gut.keys():
    model_ids = list(models_gut.keys())
    model_ids.remove(model_id)
    community_id = model_id.split("_")[0][0].upper() + model_id.split("_")[1][0].upper()+"-"
    community_ids = [community_id for id_ in model_ids]
    
    df_comm = pd.DataFrame([community_ids,model_ids])
    df_comms.append(df_comm)

# Save results
communities_gut = pd.concat(df_comms,axis=1).T
print(communities_gut[0].unique())
communities_gut.to_csv("../data/Venturelli2018/generated/communities_smetana.tsv",
                                        index=None,
                                        header=None,
                                        sep="\t")



['Full' 'BU-' 'BV-' 'CA-' 'FP-' 'EL-' 'PC-' 'CH-' 'ER-' 'DP-' 'BO-' 'BT-'
 'BH-']


In [11]:
# Media data
mets_gut = []
for rxn in env_gut.keys():
    mets_gut.append(rxn.replace("R_EX_","").replace("__91__e__93__",""))

media_name = ["LB_extend" for met in mets_gut]
media_db_gut_smetana = pd.DataFrame({"medium":media_name,"compound":mets_gut})
media_db_gut_smetana.to_csv("../data/Venturelli2018/generated/media_db_smetana.tsv",
                            index=False,
                            sep="\t")

## Prepare files for steadiercom

We need:
* Media_db (already have)
* Communities-file with relative abundance

In [12]:
with open("../data/inorganic.txt") as f:
    mets = f.read().split("\n")

mets_gut = [f"R_EX_{met}__91__e__93__" for met in mets]

with open("../data/Venturelli2018/generated/unlimited.txt",'w') as f:
    text = "\n".join(mets_gut)
    f.write(text)



In [13]:
mets_gut = [f"R_EX_{met}_e" for met in mets]

with open("../../SteadierCom/tests/data/unlimited_R_prefix.txt",'w') as f:
    text = "\n".join(mets_gut)
    f.write(text)


#### E.coli mutants

In [14]:
# NB! yhhK in data == panZ in paper
relative_abundance = pd.read_csv("../data/Guillen2021/relative_abundance.csv",index_col=0)
relative_abundance.index =relative_abundance.index.map(lambda x:x if x!="yhhK" else "panZ")

In [15]:
relative_abundance_ecoli = relative_abundance.T.mean().sort_values(ascending=False).head(7)
relative_abundance_ecoli = relative_abundance_ecoli.reset_index()
relative_abundance_ecoli["community"] = relative_abundance_ecoli["index"].map(lambda x:"Ecoli_mutants")
relative_abundance_ecoli["index"] = relative_abundance_ecoli["index"].map(lambda x:"mutant_"+x)
relative_abundance_ecoli\
        .set_index("community")\
        .reset_index()\
        .to_csv("../data/Guillen2021/generated/communities_steadiercom.tsv",
                                                     sep="\t",
                                                     header=None,
                                                     index=None)


#### Gut microbiome

**Processing datasetev3**

In [16]:
minimal_medium = pd.read_csv("../output/Venturelli2018/minimal_media.tsv",sep="\t",index_col=0)
member_id2member_name = {x.split("_")[0][0]+x.split("_")[1][0].upper():x for x in minimal_medium.columns}


datasetev3 = pd.read_excel("../data/Venturelli2018/msb178157-sup-0004-datasetev3.xlsx",sheet_name="Dataset EV3")

abundance_dict = {}
last_item = "'ER-'"
abundance_dict[last_item]=[]
for i,row in datasetev3.iterrows():
    list_ = row.iloc[0].split("    ")
    
    if len(list_)==1:
        last_item =list_[0]
        abundance_dict[last_item]=[]
    else:
        abundance_dict[last_item].append(list_[1:])

In [17]:
abundance_dataframes = {community:pd.DataFrame(abundances).applymap(lambda x: float(x)) for community,abundances in abundance_dict.items()}

for community,dataframe in abundance_dataframes.items(): dataframe.columns=["BH","CA","BU","PC","BO","BV","BT","EL","FP","CH","DP","ER"] 

for community,dataframe in abundance_dataframes.items(): dataframe.columns = dataframe.columns.map(lambda x:member_id2member_name[x])

abundance_series = {community_id.replace("'",""):dataframe.iloc[-1,:] for community_id,dataframe in abundance_dataframes.items()}
for community,series in abundance_series.items(): 
    if community!="NONE": 
        abundance_series[community]=series.drop(member_id2member_name[community[:-1]])


communities_exp_abundance = pd.concat(abundance_series)
communities_exp_abundance

ER-   Blautia_hydrogenotrophica_DSM_10507       0.0109
      Collinsella_aerofaciens_ATCC_25986        0.0255
      Bacteroides_uniformis_ATCC_8492           0.3809
      Prevotella_copri_CB7_DSM_18205            0.0003
      Bacteroides_ovatus_ATCC_8483              0.1343
                                                 ...  
NONE  Eggerthella_lenta_DSM_2243                0.0168
      Faecalibacterium_prausnitzii_A2_165       0.2393
      Clostridium_hiranonis_TO_931_DSM_13275    0.0212
      Desulfovibrio_piger_ATCC_29098            0.0019
      Eubacterium_rectale_ATCC_33656            0.0086
Name: 6, Length: 144, dtype: float64

In [18]:
for community_id in communities_exp_abundance.index.get_level_values(0).unique():
    
    new_id = community_id.replace("-","_minus")
    communities_exp_abundance[[community_id]].to_csv(f"../data/Venturelli2018/generated/communities_steadiercom/{new_id}_abundance.tsv",
                                                     sep="\t",
                                                     header=None)
    
    communities_exp_abundance[[community_id]].reset_index().iloc[:,:-1].to_csv(f"../data/Venturelli2018/generated/communities_steadiercom_no_abundance/{new_id}_abundance.tsv",
                                                     sep="\t",
                                                     header=None,
                                                     index=None)

### Maximum import rates

**Specific growth rate $\mu$**

Concentration $X_0$ (right after serial transfer) and $X_1$ (right before next transfer) in $t_0$ and $t_1$

$$
X_1 = X_0 *e^{\mu t_1} , X_0 = X_0
$$

Assuming that the cells grow back to the original concentration (after being diluted 1:20 => the cells increase in density by 21 times)

$$
X_1/X_0 = 21 = e^{\mu t} 
$$

The above equation turns into:
$$
ln(X_1/X_0) = \mu t
$$

Which means the avearage growth rate becomes
$$
\mu_{avg} = \frac{ln(21)}{t}
$$


**Substrate uptake rate $v_s$**
$$
\frac{dS}{dt} = X_0 e^{\mu t} v_s
$$

We assume that all the substrate is used between each transfer: $S_{t=end}=0$, $S_0$ is defined (and dependent on the susbtrate).

$$
dS = X_{t=0} e^{\mu t} v_s dt
$$

becomes...

$$
S_{t=end}-S_{t=0}  = X_0 * v_s * [\frac{e^{\mu t}}{\mu}]^{t=end}_{t=0}
$$

which becomes...
$$
-S_{t=0} =X_0 * v_s * \frac{e^{\mu t}-1}{\mu}
$$

from previously we have defined: $21 = e^{\mu t}$. This means that $v_s$ can be defined as...

$$
v_s = -\frac{S_{t=0} \mu}{X_{t=0} 20}
$$

In [19]:
import molmass

In [20]:
mets_id_ag = []
mets_id_bigg = []
ex_rxns_ag = []
for model in models_gut.values():
    for rxn in model.get_exchange_reactions():
        if "R_EX_" not in rxn:
            continue
        met_id_ag = model.reactions[rxn].get_substrates()[0]
        mets_id_ag.append(met_id_ag)
        met_id_bigg = model.metabolites[met_id_ag].metadata["XMLAnnotation"].split("bigg.metabolite/")[1].split("\"")[0]
        mets_id_bigg.append(met_id_bigg)
        ex_rxns_ag.append(rxn)
    
met2ex = dict(zip(mets_id_bigg,ex_rxns_ag))

In [21]:

conc = {"M_malt__91__e__93__":1, #g/L
        "M_cellb__91__e__93__":1, #g/L
        "M_pyr__91__e__93__":1, #g/L
        "M_succ__91__e__93__":0.5, #g/L
       }


molarmasses = {}
mets = conc.keys()

for met in mets:
    for model in models_gut.values():
        if met in model.metabolites:
            formula = model.metabolites[met].metadata["FORMULA"]
            molarmass = molmass.Formula(formula).mass
            molarmasses[met]=molarmass
            break
        


In [22]:
molarmasses

{'M_malt__91__e__93__': 342.297037,
 'M_cellb__91__e__93__': 342.297037,
 'M_pyr__91__e__93__': 87.054258,
 'M_succ__91__e__93__': 116.07234399999999}

In [23]:
mmolars = {}
for met,conc_i in conc.items():
    mmolars[met] = conc_i*1000/molarmasses[met]
    

In [24]:
mmolars

{'M_malt__91__e__93__': 2.9214392527739,
 'M_cellb__91__e__93__': 2.9214392527739,
 'M_pyr__91__e__93__': 11.487088891160269,
 'M_succ__91__e__93__': 4.307658334185102}

In [25]:
rxns = [met2ex[met].replace("40","91").replace("41","93") for met in media_db[media_db.medium=="LB[-O2]"].compound.values if met in met2ex.keys()]

media_db_gut["bound"]=10
media_db_gut.loc[media_db_gut.compound.isin(rxns),"bound"]=50

In [26]:
media_db_gut.to_csv("../data/Venturelli2018/generated/media_db_bounds.tsv",sep="\t",index=False)

In [27]:
media_db_gut

,medium,compound,bound
0,LB_extend,R_EX_lys_L__91__e__93__,50
1,LB_extend,R_EX_cellb__91__e__93__,10
2,LB_extend,R_EX_thymd__91__e__93__,50
3,LB_extend,R_EX_so4__91__e__93__,50
4,LB_extend,R_EX_aso3__91__e__93__,50
...,...,...,...
93,LB_extend,R_EX_ribflv__91__e__93__,50
94,LB_extend,R_EX_taur__91__e__93__,10
95,LB_extend,R_EX_adn__91__e__93__,50
96,LB_extend,R_EX_malt__91__e__93__,10


In [28]:
"R_EX_xan__91__e__93__" in media_db_gut.compound.values

True

In [29]:
media_smetana =  pd.read_csv("../data/Venturelli2018/generated/media_db_smetana.tsv",sep="\t")

In [30]:
rxns_smetana = set(["R_EX_"+met+"__91__e__93__" for met in media_smetana.compound.values])

In [31]:
rxns_gut = set(media_db_gut.compound.values)

In [32]:
rxns_smetana-rxns_gut

set()

In [33]:
rxns_old = set(pd.read_csv("/Users/idunmariaburgos/Desktop/media_db_lb_extend.tsv",sep="\t").compound.values)

In [34]:
rxns_old-rxns_gut

{'R_EX_ac__91__e__93__',
 'R_EX_gam26s__91__e__93__',
 'R_EX_glycys__91__e__93__',
 'R_EX_rbflvrd__91__e__93__'}

In [35]:
for model in models_gut.values():
    if "R_EX_ribflv__40__e__41__" in model.reactions:
        print(model.id)

M_Collinsella_aerofaciens_ATCC_25986
M_Faecalibacterium_prausnitzii_A2_165
M_Eggerthella_lenta_DSM_2243
M_Clostridium_hiranonis_TO_931_DSM_13275
M_Eubacterium_rectale_ATCC_33656
M_Blautia_hydrogenotrophica_DSM_10507


In [36]:
for model in models_gut.values():
    if "R_EX_rbflvrd__40__e__41__" in model.reactions:
        print(model.id)

M_Faecalibacterium_prausnitzii_A2_165
M_Eubacterium_rectale_ATCC_33656


In [37]:
el_test = pd.read_csv("../output/Venturelli2018/results_steadiercom/results_growth_abundance/EL_minus.tsv",sep="\t")

In [38]:
el_rxns = set([met.replace("M_","R_EX_") for met in el_test[el_test.donor=="environment"].compound.unique()])

In [39]:
el_rxns-rxns_gut

{'R_EX_26dap_R_EX__91__e__93__'}

In [40]:
el_test_env = el_test[el_test.donor=="environment"]
el_test_env[el_test_env.compound.isin([rxn.replace("R_EX_","M_") for rxn in el_rxns-rxns_gut])].groupby("compound").sum()[["mass_rate","rate","frequency"]]

,mass_rate,rate,frequency
compound,,,
M_26dap_M__91__e__93__,0.00121,0.00636,2.0


In [41]:
el_test_env[el_test_env.compound.isin([rxn.replace("R_EX_","M_") for rxn in el_rxns-rxns_gut])][["compound","frequency"]].groupby("compound").mean()

,frequency
compound,
M_26dap_M__91__e__93__,1.0


In [42]:
el_test_env[el_test_env.compound.isin([rxn.replace("R_EX_","M_") for rxn in el_rxns-rxns_gut])].groupby(["receiver","compound"]).sum()[["mass_rate","rate","frequency"]]

,,mass_rate,rate,frequency
receiver,compound,,,
Bacteroides_vulgatus_ATCC_8482,M_26dap_M__91__e__93__,0.001036,0.005447,1.0
Collinsella_aerofaciens_ATCC_25986,M_26dap_M__91__e__93__,0.000174,0.000913,1.0


In [43]:
for model in models_gut.values():
    if "M_4hpro_LT__91__e__93__" in model.metabolites:
        print(model.metabolites["M_4hpro_LT__91__e__93__"])
        print(model.id)

trans-4-hydroxy-L-proline
M_Clostridium_hiranonis_TO_931_DSM_13275


In [44]:
for model in models_gut.values():
    if "M_26dap_M__91__e__93__" in model.metabolites:
        print(model.metabolites["M_26dap_M__91__e__93__"])
        print(model.id)

meso-2,6-Diaminoheptanedioate
M_Bacteroides_vulgatus_ATCC_8482
meso-2,6-Diaminoheptanedioate
M_Collinsella_aerofaciens_ATCC_25986


In [45]:
for model in models_gut.values():
    if "M_gam26s__91__e__93__" in model.metabolites:
        print(model.metabolites["M_gam26s__91__e__93__"])
        print(model.id)

Glucosamine 2,6-sulfate
M_Bacteroides_ovatus_ATCC_8483
Glucosamine 2,6-sulfate
M_Bacteroides_thetaiotaomicron_VPI_5482
